# ACV Data Inspection

This notebook examines the structure, schema, data quality, missingness,
sampling behaviour, and telemetry characteristics of the ACV training and
test datasets.

The objective is to identify a consistent set of signals that can support
refrigerant-leakage fault localisation across the eight train cars.

## 1. Dataset Setup

Define relative paths to the ACV training data, test data, and training labels.

Relative paths are used instead of machine-specific absolute paths so that the notebook remains portable across different development environments.

In [4]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Training Labels

Load the provided training labels to identify the faulty car associated with each training case.

These labels provide the ground truth that will later be used for model development and validation.

In [5]:
DATA_DIR = Path("../../../../data/ACV")

TRAIN_DIR = DATA_DIR / "Train"
TEST_DIR = DATA_DIR / "Test"
LABELS_PATH = DATA_DIR / "Train_Labels.csv"

train_files = sorted(TRAIN_DIR.glob("*.xlsx"))
test_files = sorted(TEST_DIR.glob("*.xlsx"))

print("Training files:", len(train_files))
print("Test files:", len(test_files))

for file in train_files:
    print(file.name)

Training files: 6
Test files: 1
acv_case_01.xlsx
acv_case_02.xlsx
acv_case_03.xlsx
acv_case_04.xlsx
acv_case_05.xlsx
acv_case_06.xlsx


## 3. Initial Inspection of a Training Case

Begin by examining a single training case to understand the basic structure of the ACV telemetry before comparing all cases.

Case 01 is used as the initial example.

In [6]:
labels = pd.read_csv(LABELS_PATH)

labels

,filename,faulty_car
0,acv_case_01.xlsx,1
1,acv_case_02.xlsx,2
2,acv_case_03.xlsx,3
3,acv_case_04.xlsx,1
4,acv_case_05.xlsx,4
5,acv_case_06.xlsx,6


In [7]:
case_01 = pd.read_excel(train_files[0])

print("Shape:", case_01.shape)

case_01.head()

Shape: (6999, 67)


,Car model,Train number,Time,Car 08 - ACV Setting Mode,Car 01 - ACV Control Temperature (Cooling),Car 04 - ACV Setting Mode,Car 01 - ACV Control Temperature (Heating),Car 08 - ACV Running Mode,Car 02 - Outdoor Average Temperature,Car 02 - ACV Setting Mode,...,Car 05 - ACV Control Temperature (Heating),Car 07 - ACV Setting Mode,Car 04 - ACV Control Temperature (Heating),Car 04 - Indoor Average Temperature,Car 02 - ACV Control Temperature (Cooling),Car 05 - ACV Setting Mode,Car 08 - Outdoor Average Temperature,Car 01 - ACV Running Mode,Car 04 - Outdoor Average Temperature,Car 06 - ACV Control Temperature (Cooling)
0,A,620,2023-05-18 00:00:00,Centralized Control,25.0,Centralized Control,20.0,Automatic Cooling,26.0,Centralized Control,...,20.0,Centralized Control,20.0,24.5,25.0,Centralized Control,26.0,Automatic Cooling,27.0,25.0
1,A,620,2023-05-18 00:00:30,Centralized Control,25.0,Centralized Control,20.0,Automatic Cooling,26.0,Centralized Control,...,20.0,Centralized Control,20.0,24.5,25.0,Centralized Control,26.0,Automatic Cooling,27.0,25.0
2,A,620,2023-05-18 00:01:00,Centralized Control,25.0,Centralized Control,20.0,Automatic Cooling,26.0,Centralized Control,...,20.0,Centralized Control,20.0,24.5,25.0,Centralized Control,26.0,Automatic Cooling,27.0,25.0
3,A,620,2023-05-18 00:01:30,Centralized Control,25.0,Centralized Control,20.0,Automatic Cooling,26.0,Centralized Control,...,20.0,Centralized Control,20.0,24.5,25.0,Centralized Control,26.0,Automatic Cooling,27.0,25.0
4,A,620,2023-05-18 00:02:00,Centralized Control,25.0,Centralized Control,20.0,Automatic Cooling,26.0,Centralized Control,...,20.0,Centralized Control,20.0,24.5,25.0,Centralized Control,26.0,Automatic Cooling,27.0,25.0


### 3.1 Data Types and Completeness

Inspect the columns, data types, and non-null counts of Case 01.

This provides an initial indication of how the telemetry is represented and whether missing data may require additional handling.

In [8]:
case_01.info()

<class 'pandas.DataFrame'>
RangeIndex: 6999 entries, 0 to 6998
Data columns (total 67 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   Car model                                   6999 non-null   str           
 1   Train number                                6999 non-null   int64         
 2   Time                                        6999 non-null   datetime64[us]
 3   Car 08 - ACV Setting Mode                   6322 non-null   str           
 4   Car 01 - ACV Control Temperature (Cooling)  6322 non-null   float64       
 5   Car 04 - ACV Setting Mode                   6322 non-null   str           
 6   Car 01 - ACV Control Temperature (Heating)  6322 non-null   float64       
 7   Car 08 - ACV Running Mode                   6322 non-null   str           
 8   Car 02 - Outdoor Average Temperature        6322 non-null   float64       
 9   Car 02 - ACV Settin

### 3.2 Column Structure

Inspect the complete list of columns to understand how metadata, car identifiers, and ACV parameters are encoded in the dataset.

In [9]:
case_01.columns.tolist()

['Car model',
 'Train number',
 'Time',
 'Car 08 - ACV Setting Mode',
 'Car 01 - ACV Control Temperature (Cooling)',
 'Car 04 - ACV Setting Mode',
 'Car 01 - ACV Control Temperature (Heating)',
 'Car 08 - ACV Running Mode',
 'Car 02 - Outdoor Average Temperature',
 'Car 02 - ACV Setting Mode',
 'Car 07 - ACV Running Mode',
 'Car 07 - Outdoor Average Temperature',
 'Car 08 - ACV Control Temperature (Heating)',
 'Car 08 - ACV Information Valid',
 'Car 07 - ACV Information Valid',
 'Car 06 - ACV Information Valid',
 'Car 03 - Outdoor Average Temperature',
 'Car 05 - ACV Running Mode',
 'Car 06 - ACV Setting Mode',
 'Car 03 - ACV Control Temperature (Cooling)',
 'Car 06 - Load Halved',
 'Car 05 - Indoor Average Temperature',
 'Car 04 - ACV Control Temperature (Cooling)',
 'Car 03 - ACV Running Mode',
 'Car 07 - ACV Control Temperature (Heating)',
 'Car 03 - ACV Setting Mode',
 'Car 08 - Load Halved',
 'Car 01 - Load Halved',
 'Car 02 - ACV Control Temperature (Heating)',
 'Car 08 - ACV Con

### 3.3 Metadata and Telemetry Columns

Separate the general train metadata from the car-specific ACV telemetry.

The telemetry columns follow the naming convention `Car XX - <parameter>`, allowing the car identifier and parameter name to be extracted programmatically.

In [10]:
metadata_columns = ["Car model", "Train number", "Time"]

telemetry_columns = [
    col for col in case_01.columns
    if col not in metadata_columns
]

len(telemetry_columns)

64

### 3.4 Car Discovery

Extract the unique car identifiers directly from the telemetry column names.

Discovering cars dynamically avoids relying on fixed column positions and makes the ingestion process more robust to differences in column ordering.

In [11]:
cars = sorted({
    col.split(" - ", 1)[0]
    for col in telemetry_columns
})

cars

['Car 01',
 'Car 02',
 'Car 03',
 'Car 04',
 'Car 05',
 'Car 06',
 'Car 07',
 'Car 08']

### 3.5 Parameter Discovery

Extract the unique ACV parameter names from the telemetry columns.

This determines which measurements are available in Case 01 and provides a basis for comparing schemas across the remaining training cases.

In [12]:
parameters = sorted({
    col.split(" - ", 1)[1]
    for col in telemetry_columns
})

print(f"Number of parameters: {len(parameters)}")

for parameter in parameters:
    print(parameter)

Number of parameters: 8
ACV Control Temperature (Cooling)
ACV Control Temperature (Heating)
ACV Information Valid
ACV Running Mode
ACV Setting Mode
Indoor Average Temperature
Load Halved
Outdoor Average Temperature


### 3.6 Per-Car Schema Validation

Verify the number of telemetry parameters available for each car.

A consistent parameter count within a case indicates that all cars are represented using the same telemetry schema. This check will later be extended across all training cases to identify differences between files.

In [13]:
for car in cars:
    car_columns = [
        col for col in telemetry_columns
        if col.startswith(f"{car} - ")
    ]

    print(f"{car}: {len(car_columns)} parameters")

Car 01: 8 parameters
Car 02: 8 parameters
Car 03: 8 parameters
Car 04: 8 parameters
Car 05: 8 parameters
Car 06: 8 parameters
Car 07: 8 parameters
Car 08: 8 parameters


**Observation:** Case 01 contains eight cars, with eight ACV telemetry parameters available for each car. The telemetry columns are not necessarily grouped by car, so subsequent processing should identify cars and parameters using the column names rather than fixed column positions.

## 4. Comparison Across Training Cases

The previous section inspected Case 01 individually to understand the basic structure of the ACV telemetry.

The same inspection is now applied programmatically across all training cases. This allows differences in dataset size, number of cars, and telemetry schema to be identified without making assumptions about individual files.

In [14]:
case_summaries = []

for file in train_files:
    df = pd.read_excel(file)

    # Separate metadata from telemetry
    telemetry_cols = [
        col for col in df.columns
        if col not in metadata_columns
    ]

    # Discover cars dynamically
    case_cars = sorted({
        col.split(" - ", 1)[0]
        for col in telemetry_cols
        if " - " in col
    })

    # Discover parameters dynamically
    case_parameters = sorted({
        col.split(" - ", 1)[1]
        for col in telemetry_cols
        if " - " in col
    })

    case_summaries.append({
        "File": file.name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Cars": len(case_cars),
        "Unique Parameters": len(case_parameters)
    })

case_summary_df = pd.DataFrame(case_summaries)

case_summary_df

,File,Rows,Columns,Cars,Unique Parameters
0,acv_case_01.xlsx,6999,67,8,8
1,acv_case_02.xlsx,9187,67,8,8
2,acv_case_03.xlsx,8310,67,8,8
3,acv_case_04.xlsx,22262,483,8,63
4,acv_case_05.xlsx,6972,67,8,8
5,acv_case_06.xlsx,3263,67,8,8


### 4.1 Observations

The six training cases contain the same eight train cars but differ in recording length.

Five cases (01, 02, 03, 05, and 06) contain 67 columns, consisting of three metadata columns and 64 telemetry columns corresponding to eight parameters for each of the eight cars.

Case 04 is a significant schema outlier. It contains 483 columns and 63 unique ACV parameters, substantially more than the other training cases. Its recording is also considerably longer, with 22,262 rows.

Therefore, the ACV processing pipeline should not assume a fixed number of rows, columns, or telemetry parameters. Schema discovery and parameter handling should be performed dynamically for each input file.

## 5. Schema Comparison Across Training Cases

Case 04 contains substantially more telemetry parameters than the other training cases.

To understand this schema difference, the parameter sets of each case are extracted and compared. This identifies parameters shared across cases and parameters that are only available in the expanded Case 04 schema.

In [15]:
case_parameter_sets = {}

for file in train_files:
    df = pd.read_excel(file)

    telemetry_cols = [
        col for col in df.columns
        if col not in metadata_columns and " - " in col
    ]

    case_parameters = {
        col.split(" - ", 1)[1]
        for col in telemetry_cols
    }

    case_parameter_sets[file.name] = case_parameters

for filename, params in case_parameter_sets.items():
    print(f"{filename}: {len(params)} parameters")

acv_case_01.xlsx: 8 parameters
acv_case_02.xlsx: 8 parameters
acv_case_03.xlsx: 8 parameters
acv_case_04.xlsx: 63 parameters
acv_case_05.xlsx: 8 parameters
acv_case_06.xlsx: 8 parameters


### 5.1 Common Parameters

Identify the telemetry parameters that are available in every training case.

Parameters shared across all cases are particularly important because they provide a consistent feature space that can potentially be used when developing a model that must generalise across different input schemas.

In [16]:
common_parameters = set.intersection(*case_parameter_sets.values())

print(f"Parameters common to all cases: {len(common_parameters)}")

for parameter in sorted(common_parameters):
    print(parameter)

Parameters common to all cases: 1
ACV Running Mode


**Observation:** Only `ACV Running Mode` appears under the exact same parameter name across all six training cases. Further inspection shows that Cases 01, 02, 03, 05, and 06 have nearly identical schemas, with the only observed naming difference being the outdoor temperature measurement (`Outdoor Average Temperature` versus `Outside Temperature Sensor Reading`).

The low intersection across all six cases is therefore primarily caused by the substantially different schema of Case 04 rather than widespread differences among the standard-sized cases.

### 5.2 Parameter Sets by Training Case

Inspect the parameter names available in each training case.

Although five cases contain eight unique parameters each, the low intersection across all six cases suggests that the parameter names or available measurements differ between files. Examining each parameter set helps determine whether these differences represent genuinely different signals or naming variations.

In [17]:
for filename, params in case_parameter_sets.items():
    print(f"\n{filename}")
    print("-" * len(filename))

    for parameter in sorted(params):
        print(parameter)


acv_case_01.xlsx
----------------
ACV Control Temperature (Cooling)
ACV Control Temperature (Heating)
ACV Information Valid
ACV Running Mode
ACV Setting Mode
Indoor Average Temperature
Load Halved
Outdoor Average Temperature

acv_case_02.xlsx
----------------
ACV Control Temperature (Cooling)
ACV Control Temperature (Heating)
ACV Information Valid
ACV Running Mode
ACV Setting Mode
Indoor Average Temperature
Load Halved
Outdoor Average Temperature

acv_case_03.xlsx
----------------
ACV Control Temperature (Cooling)
ACV Control Temperature (Heating)
ACV Information Valid
ACV Running Mode
ACV Setting Mode
Indoor Average Temperature
Load Halved
Outdoor Average Temperature

acv_case_04.xlsx
----------------
ACV Control Mode
ACV Grounding Detection Status
ACV Grounding Test Command
ACV Operating Mode
ACV Running Mode
ACV Self-Check
ACV Standby Startup Command
ACV Time-Staggered Startup
Auxiliary Compressor Control Circuit Breaker
Auxiliary Compressor Control Relay
Auxiliary Compressor Start

### 5.3 Comparison of Standard-Sized Schemas

Compare the five training cases that each contain eight telemetry parameters.

This isolates naming and schema differences among the similarly sized cases before considering the much larger Case 04 schema.

In [18]:
standard_cases = [
    "acv_case_01.xlsx",
    "acv_case_02.xlsx",
    "acv_case_03.xlsx",
    "acv_case_05.xlsx",
    "acv_case_06.xlsx"
]

for filename in standard_cases:
    print(f"\n{filename}")

    for parameter in sorted(case_parameter_sets[filename]):
        print(f"  - {parameter}")


acv_case_01.xlsx
  - ACV Control Temperature (Cooling)
  - ACV Control Temperature (Heating)
  - ACV Information Valid
  - ACV Running Mode
  - ACV Setting Mode
  - Indoor Average Temperature
  - Load Halved
  - Outdoor Average Temperature

acv_case_02.xlsx
  - ACV Control Temperature (Cooling)
  - ACV Control Temperature (Heating)
  - ACV Information Valid
  - ACV Running Mode
  - ACV Setting Mode
  - Indoor Average Temperature
  - Load Halved
  - Outdoor Average Temperature

acv_case_03.xlsx
  - ACV Control Temperature (Cooling)
  - ACV Control Temperature (Heating)
  - ACV Information Valid
  - ACV Running Mode
  - ACV Setting Mode
  - Indoor Average Temperature
  - Load Halved
  - Outdoor Average Temperature

acv_case_05.xlsx
  - ACV Control Temperature (Cooling)
  - ACV Control Temperature (Heating)
  - ACV Information Valid
  - ACV Running Mode
  - ACV Setting Mode
  - Indoor Average Temperature
  - Load Halved
  - Outside Temperature Sensor Reading

acv_case_06.xlsx
  - ACV Con

**Observation:** Cases 01, 02, 03, 05, and 06 share seven identically named ACV parameters. Their eighth parameter represents outdoor temperature but uses two different names: `Outdoor Average Temperature` in Cases 01–03 and `Outside Temperature Sensor Reading` in Cases 05–06.

These two parameter names may represent equivalent or closely related measurements. This should be verified from their values and behaviour before they are standardised under a common parameter name during preprocessing.

### 5.4 Inspection of the Expanded Case 04 Schema

Case 04 differs substantially from the other training cases, containing 63 unique telemetry parameters per car instead of eight.

The complete parameter set is inspected to determine whether Case 04 contains equivalent measurements under different names, additional diagnostic signals, or a fundamentally different telemetry representation.

In [19]:
case_04_parameters = sorted(
    case_parameter_sets["acv_case_04.xlsx"]
)

print(f"Case 04 parameters: {len(case_04_parameters)}\n")

for parameter in case_04_parameters:
    print(parameter)

Case 04 parameters: 63

ACV Control Mode
ACV Grounding Detection Status
ACV Grounding Test Command
ACV Operating Mode
ACV Running Mode
ACV Self-Check
ACV Standby Startup Command
ACV Time-Staggered Startup
Auxiliary Compressor Control Circuit Breaker
Auxiliary Compressor Control Relay
Auxiliary Compressor Start Switch
Cab Compressor Running
Cab Condenser Fan Running
Cab Ventilation Fan Running
Car Wash Signal
Compressor 1 Fault
Compressor 1 Running
Compressor 2 Fault
Compressor 2 Running
Condenser Fan 1 Fault
Condenser Fan 1 Running
Condenser Fan 2 Fault
Condenser Fan 2 Running
Depot Servicing Mode Command
Duct Electric Heating Running
Electric Heater 1 Fault
Electric Heater 1 Running
Electric Heater 2 Fault
Electric Heater 2 Running
Emergency Ventilation Inverter Running
End 1 Vestibule Electric Heating Running
End 2 Vestibule Electric Heating Running
End-Change Signal
Exhaust Damper Closed
Exhaust Fan Fault
Exhaust Fan Running
Exhaust Pressure Wave Valve Closed
Fresh Air Temperature D

**Observation:** Case 04 provides a substantially richer ACV diagnostic schema than the other training cases. Its 63 parameters include detailed equipment operating states, fault indicators, refrigeration-system pressure measurements, compressor and fan states, temperature measurements, control modes, and command signals.

Several Case 04 parameters appear conceptually related to measurements in the standard eight-parameter schema, such as passenger-cabin/fresh-air temperatures and target temperature values. However, the naming and level of detail differ considerably.

Case 04 also contains potentially fault-relevant signals that are unavailable in the other training cases, including refrigeration-system high- and low-pressure measurements. These case-specific signals should not automatically be used as required model inputs because their availability in other cases and the unseen test case is not guaranteed.

## 6. Test Dataset Schema

Inspect the structure of the provided ACV test case.

The test schema is important for determining which telemetry parameters are available during final inference. Features used by the prediction pipeline must either be available in the test data or be handled gracefully when absent.

In [20]:
test_df = pd.read_excel(test_files[0])

print("Test file:", test_files[0].name)
print("Rows:", test_df.shape[0])
print("Columns:", test_df.shape[1])

Test file: acv_test_case.xlsx
Rows: 9082
Columns: 67


### 6.1 Test Cars and Parameters

Extract the cars and telemetry parameters available in the test case using the same dynamic schema-discovery approach applied to the training data.

In [21]:
test_telemetry_cols = [
    col for col in test_df.columns
    if col not in metadata_columns and " - " in col
]

test_cars = sorted({
    col.split(" - ", 1)[0]
    for col in test_telemetry_cols
})

test_parameters = sorted({
    col.split(" - ", 1)[1]
    for col in test_telemetry_cols
})

print(f"Cars: {len(test_cars)}")
print(f"Parameters: {len(test_parameters)}")

print("\nParameters:")
for parameter in test_parameters:
    print(parameter)

Cars: 8
Parameters: 8

Parameters:
ACV Control Temperature (Cooling)
ACV Control Temperature (Heating)
ACV Information Valid
ACV Running Mode
ACV Setting Mode
Indoor Average Temperature
Load Halved
Outdoor Average Temperature


### 6.2 Observation

The test case contains 8 cars and 8 telemetry parameters per car, with a total of 67 columns. Its schema exactly matches the standard schema used by training Cases 01, 02, and 03.

Cases 05 and 06 use the same general eight-parameter structure but represent the outdoor temperature measurement using `Outside Temperature Sensor Reading` instead of `Outdoor Average Temperature`.

Case 04 contains a substantially richer 63-parameter schema. Since these additional parameters are unavailable in the test case, the final prediction pipeline cannot depend on Case-04-specific signals such as refrigeration-system pressure measurements or detailed compressor and fan states.

Therefore, model development should prioritise features that can be derived consistently from the standard eight-parameter telemetry available in the test case, while handling equivalent parameter names across training cases during preprocessing.

### 6.3 Schema Implication for Model Development

Based on the training and test schemas, the model should prioritise telemetry that can be represented consistently across the standard ACV cases and the test case.

```text
Cases 01–03                 Cases 05–06
8 parameters                8 parameters
Outdoor Average Temp        Outside Temp Sensor
      │                            │
      └────────────┬───────────────┘
                   ↓
          Standardised Schema
                   ↑
                   │
              Case 04
           63 parameters
       (requires mapping /
        compatible subset)
                   │
                   ↓
                Features
                   │
                   ↓
              Fault Model
                   │
                   ↓
               Test Case
             8 parameters
```

The final feature pipeline should not depend on Case-04-specific parameters because these signals are unavailable in the test dataset. Instead, Case 04 should only contribute information that can be mapped to the standard feature representation.

## 7. Data Quality Inspection

Before preprocessing and feature engineering, the training and test datasets are inspected for missing values and data-type inconsistencies.

This helps identify whether telemetry signals require cleaning, conversion, or special handling before they can be used for analysis and modelling.

In [22]:
data_quality_summary = []

all_files = train_files + test_files

for file in all_files:
    df = pd.read_excel(file)

    total_values = df.shape[0] * df.shape[1]
    missing_values = df.isna().sum().sum()

    data_quality_summary.append({
        "File": file.name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": missing_values,
        "Missing (%)": (missing_values / total_values) * 100
    })

data_quality_df = pd.DataFrame(data_quality_summary)

data_quality_df

,File,Rows,Columns,Missing Values,Missing (%)
0,acv_case_01.xlsx,6999,67,43328,9.239699
1,acv_case_02.xlsx,9187,67,51520,8.370036
2,acv_case_03.xlsx,8310,67,0,0.000000
3,acv_case_04.xlsx,22262,483,5442413,50.615110
4,acv_case_05.xlsx,6972,67,43136,9.234379
5,acv_case_06.xlsx,3263,67,28672,13.114934
6,acv_test_case.xlsx,9082,67,68352,11.232978


### 7.1 Missing Value Summary

Missing values are present in most ACV files, although their prevalence varies considerably between cases.

Case 03 contains no missing values, while the standard-schema Cases 01, 02, 05, and 06 contain approximately 8–13% missing values. The test case similarly contains approximately 11% missing values, indicating that missing-data handling will be required during final inference.

Case 04 is a significant outlier, with approximately 51% of all values missing. Given its expanded 63-parameter telemetry schema, this may indicate that many diagnostic signals are only populated under particular operating conditions rather than representing random data loss.

Therefore, rows containing missing values should not be removed indiscriminately. The location and pattern of missingness must first be investigated at the parameter and car levels before selecting an appropriate preprocessing strategy.

### 7.2 Missing Values by Parameter

To determine whether missing values are concentrated in particular telemetry signals, missingness is analysed at the parameter level for the standard-schema training cases and the test case.

Because each parameter is recorded separately for eight cars, the missing-value rate is aggregated across all columns corresponding to the same parameter.

In [23]:
standard_files = [
    TRAIN_DIR / "acv_case_01.xlsx",
    TRAIN_DIR / "acv_case_02.xlsx",
    TRAIN_DIR / "acv_case_03.xlsx",
    TRAIN_DIR / "acv_case_05.xlsx",
    TRAIN_DIR / "acv_case_06.xlsx",
    TEST_DIR / "acv_test_case.xlsx"
]

parameter_missing_summary = []

for file in standard_files:
    df = pd.read_excel(file)

    telemetry_cols = [
        col for col in df.columns
        if col not in metadata_columns and " - " in col
    ]

    parameters_in_file = sorted({
        col.split(" - ", 1)[1]
        for col in telemetry_cols
    })

    for parameter in parameters_in_file:
        parameter_cols = [
            col for col in telemetry_cols
            if col.split(" - ", 1)[1] == parameter
        ]

        parameter_data = df[parameter_cols]

        total_values = parameter_data.size
        missing_values = parameter_data.isna().sum().sum()

        parameter_missing_summary.append({
            "File": file.name,
            "Parameter": parameter,
            "Missing (%)": (missing_values / total_values) * 100
        })

parameter_missing_df = pd.DataFrame(parameter_missing_summary)

parameter_missing_df

,File,Parameter,Missing (%)
0,acv_case_01.xlsx,ACV Control Temperature (Cooling),9.672810
1,acv_case_01.xlsx,ACV Control Temperature (Heating),9.672810
2,acv_case_01.xlsx,ACV Information Valid,9.672810
3,acv_case_01.xlsx,ACV Running Mode,9.672810
4,acv_case_01.xlsx,ACV Setting Mode,9.672810
5,acv_case_01.xlsx,Indoor Average Temperature,9.672810
6,acv_case_01.xlsx,Load Halved,9.672810
7,acv_case_01.xlsx,Outdoor Average Temperature,9.672810
8,acv_case_02.xlsx,ACV Control Temperature (Cooling),8.762382
9,acv_case_02.xlsx,ACV Control Temperature (Heating),8.762382


### 7.3 Missing-Value Matrix

The parameter-level missingness results are reshaped into a matrix to make patterns across cases easier to compare.

In [24]:
missing_matrix = parameter_missing_df.pivot(
    index="Parameter",
    columns="File",
    values="Missing (%)"
)

missing_matrix.round(2)

File,acv_case_01.xlsx,acv_case_02.xlsx,acv_case_03.xlsx,acv_case_05.xlsx,acv_case_06.xlsx,acv_test_case.xlsx
Parameter,,,,,,
ACV Control Temperature (Cooling),9.67,8.76,0.0,9.67,13.73,11.76
ACV Control Temperature (Heating),9.67,8.76,0.0,9.67,13.73,11.76
ACV Information Valid,9.67,8.76,0.0,9.67,13.73,11.76
ACV Running Mode,9.67,8.76,0.0,9.67,13.73,11.76
ACV Setting Mode,9.67,8.76,0.0,9.67,13.73,11.76
Indoor Average Temperature,9.67,8.76,0.0,9.67,13.73,11.76
Load Halved,9.67,8.76,0.0,9.67,13.73,11.76
Outdoor Average Temperature,9.67,8.76,0.0,NaN,NaN,11.76
Outside Temperature Sensor Reading,NaN,NaN,NaN,9.67,13.73,NaN


### 7.4 Observation

For each standard-schema file, all available ACV parameters have the same missing-value percentage. For example, every parameter in Case 01 has approximately 9.67% missing values, while every parameter in the test case has approximately 11.76%.

This suggests that missingness is not concentrated in specific telemetry parameters. Instead, the missing-data pattern may occur at the car level or across particular time periods. Further inspection is required to determine the structure of the missingness before selecting a preprocessing strategy.

The `NaN` entries in the comparison matrix for the outdoor-temperature parameters indicate schema absence rather than missing observations. Cases 01–03 and the test case use `Outdoor Average Temperature`, whereas Cases 05–06 use `Outside Temperature Sensor Reading`.

## 8. Missing Values by Car

Since missingness is approximately equal across parameters within each file, the next step is to determine whether missing values are concentrated in particular cars.

The missing-value percentage is calculated across all telemetry parameters belonging to each car.

In [25]:
car_missing_summary = []

for file in standard_files:
    df = pd.read_excel(file)

    telemetry_cols = [
        col for col in df.columns
        if col not in metadata_columns and " - " in col
    ]

    cars_in_file = sorted({
        col.split(" - ", 1)[0]
        for col in telemetry_cols
    })

    for car in cars_in_file:
        car_cols = [
            col for col in telemetry_cols
            if col.startswith(f"{car} - ")
        ]

        car_data = df[car_cols]

        total_values = car_data.size
        missing_values = car_data.isna().sum().sum()

        car_missing_summary.append({
            "File": file.name,
            "Car": car,
            "Missing (%)": (missing_values / total_values) * 100
        })

car_missing_df = pd.DataFrame(car_missing_summary)

car_missing_matrix = car_missing_df.pivot(
    index="Car",
    columns="File",
    values="Missing (%)"
)

car_missing_matrix.round(2)

File,acv_case_01.xlsx,acv_case_02.xlsx,acv_case_03.xlsx,acv_case_05.xlsx,acv_case_06.xlsx,acv_test_case.xlsx
Car,,,,,,
Car 01,9.67,8.76,0.0,9.67,13.73,11.76
Car 02,9.67,8.76,0.0,9.67,13.73,11.76
Car 03,9.67,8.76,0.0,9.67,13.73,11.76
Car 04,9.67,8.76,0.0,9.67,13.73,11.76
Car 05,9.67,8.76,0.0,9.67,13.73,11.76
Car 06,9.67,8.76,0.0,9.67,13.73,11.76
Car 07,9.67,8.76,0.0,9.67,13.73,11.76
Car 08,9.67,8.76,0.0,9.67,13.73,11.76


### 8.1 Observation

All eight cars within each standard-schema file have identical missing-value percentages. Therefore, missingness is not concentrated in any particular car.

Combined with the previous parameter-level analysis, where all parameters within a file also exhibited identical missing-value rates, this suggests that missing observations may occur simultaneously across multiple cars and parameters at particular timestamps.

The temporal pattern of missingness should therefore be inspected next.

## 9. Temporal Pattern of Missing Values

The previous analyses show that missingness is distributed uniformly across both telemetry parameters and cars within each file.

To determine whether missing observations occur simultaneously, the number of missing telemetry values is calculated for every timestamp. This reveals whether the missing data consists of isolated values or entire periods of unavailable telemetry.

In [26]:
row_missing_summary = []

for file in standard_files:
    df = pd.read_excel(file)

    telemetry_cols = [
        col for col in df.columns
        if col not in metadata_columns and " - " in col
    ]

    missing_per_row = df[telemetry_cols].isna().sum(axis=1)

    row_missing_summary.append({
        "File": file.name,
        "Rows": len(df),
        "Rows with Missing Data": (missing_per_row > 0).sum(),
        "Fully Missing Telemetry Rows": (
            missing_per_row == len(telemetry_cols)
        ).sum(),
        "Partially Missing Telemetry Rows": (
            (missing_per_row > 0)
            & (missing_per_row < len(telemetry_cols))
        ).sum()
    })

row_missing_df = pd.DataFrame(row_missing_summary)

row_missing_df

,File,Rows,Rows with Missing Data,Fully Missing Telemetry Rows,Partially Missing Telemetry Rows
0,acv_case_01.xlsx,6999,677,677,0
1,acv_case_02.xlsx,9187,805,805,0
2,acv_case_03.xlsx,8310,0,0,0
3,acv_case_05.xlsx,6972,674,674,0
4,acv_case_06.xlsx,3263,448,448,0
5,acv_test_case.xlsx,9082,1068,1068,0


### 9.1 Observation

All rows containing missing telemetry are fully missing across the 64 standard ACV telemetry columns. No partially missing telemetry rows were observed in any of the standard-schema training cases or the test case.

This confirms that the missingness occurs at the timestamp level rather than being isolated to individual cars or parameters. In other words, when telemetry is unavailable, the measurements for all eight cars and all eight ACV parameters are absent simultaneously.

This simplifies subsequent preprocessing because individual sensor values do not require independent imputation. However, the temporal distribution of these fully missing periods should still be examined before deciding whether to remove or reconstruct them.

## 10. Missing Telemetry Gap Analysis

The missing telemetry rows occur simultaneously across all cars and parameters. The next step is to examine whether these missing rows occur as isolated timestamps or as consecutive gaps.

Understanding the duration and frequency of missing periods is important before removing or reconstructing observations, particularly because subsequent feature engineering may use temporal behaviour.

In [27]:
gap_summary = []

for file in standard_files:
    df = pd.read_excel(file)

    telemetry_cols = [
        col for col in df.columns
        if col not in metadata_columns and " - " in col
    ]

    # True when the entire telemetry row is missing
    missing_rows = df[telemetry_cols].isna().all(axis=1)

    # Assign an ID whenever missing/non-missing status changes
    groups = missing_rows.ne(missing_rows.shift()).cumsum()

    # Count consecutive missing rows
    missing_gap_lengths = (
        missing_rows[missing_rows]
        .groupby(groups[missing_rows])
        .size()
    )

    gap_summary.append({
        "File": file.name,
        "Number of Missing Gaps": len(missing_gap_lengths),
        "Shortest Gap (rows)": (
            missing_gap_lengths.min()
            if len(missing_gap_lengths) > 0 else 0
        ),
        "Median Gap (rows)": (
            missing_gap_lengths.median()
            if len(missing_gap_lengths) > 0 else 0
        ),
        "Longest Gap (rows)": (
            missing_gap_lengths.max()
            if len(missing_gap_lengths) > 0 else 0
        )
    })

gap_summary_df = pd.DataFrame(gap_summary)

gap_summary_df

,File,Number of Missing Gaps,Shortest Gap (rows),Median Gap (rows),Longest Gap (rows)
0,acv_case_01.xlsx,332,1,2.0,10
1,acv_case_02.xlsx,312,1,1.0,35
2,acv_case_03.xlsx,0,0,0.0,0
3,acv_case_05.xlsx,183,1,3.0,35
4,acv_case_06.xlsx,137,1,2.0,54
5,acv_test_case.xlsx,256,1,3.0,43


### 10.1 Observation

Missing telemetry is distributed across multiple gaps rather than occurring as a single missing block.

Most gaps are relatively short, with median gap lengths ranging from one to three rows across the files containing missing telemetry. However, longer gaps are also present, reaching 54 consecutive rows in Case 06 and 43 consecutive rows in the test case.

Therefore, the missing observations should not yet be treated as simple isolated values. Before deciding whether to remove or reconstruct these periods, the timestamp structure and actual sampling intervals should be examined.

## 11. Timestamp and Sampling Interval Inspection

The ACV data is time-series telemetry, so the `Time` column is inspected to determine its representation, ordering, and sampling interval.

This is necessary for interpreting the duration of missing telemetry gaps and for later temporal feature engineering.

In [28]:
time_summary = []

for file in all_files:
    df = pd.read_excel(file)

    time_col = df["Time"]

    time_summary.append({
        "File": file.name,
        "Time dtype": str(time_col.dtype),
        "Missing Time Values": time_col.isna().sum(),
        "First Time": time_col.iloc[0],
        "Last Time": time_col.iloc[-1]
    })

time_summary_df = pd.DataFrame(time_summary)

time_summary_df

,File,Time dtype,Missing Time Values,First Time,Last Time
0,acv_case_01.xlsx,datetime64[us],0,2023-05-18 00:00:00,2023-05-21 20:49:30
1,acv_case_02.xlsx,datetime64[us],0,2020-07-09 00:00:00,2020-07-12 23:35:00
2,acv_case_03.xlsx,datetime64[us],0,2021-09-30 05:16:00,2021-10-03 23:59:30
3,acv_case_04.xlsx,datetime64[us],0,2023-08-15 07:06:50,2023-08-18 20:41:00
4,acv_case_05.xlsx,datetime64[us],0,2021-03-03 06:19:00,2021-03-06 23:59:30
5,acv_case_06.xlsx,datetime64[us],0,2020-07-06 08:42:00,2020-07-08 23:59:30
6,acv_test_case.xlsx,datetime64[us],0,2021-06-24 00:00:00,2021-06-27 23:59:30


### 11.1 Observation

The `Time` column is successfully represented as a datetime value in all training and test files, with no missing timestamps.

Each file contains several days of continuous timestamped observations. Since the timestamps are already stored as datetime values, sampling intervals can be calculated directly using consecutive timestamp differences.

The sampling interval should be validated separately for each file because Case 04 uses a substantially different telemetry schema and may also differ in sampling frequency.

### 11.2 Sampling Interval Analysis

Calculate the time difference between consecutive observations in each file to determine the actual sampling frequency and identify any irregular timestamp gaps.

In [29]:
sampling_summary = []

for file in all_files:
    df = pd.read_excel(file)

    time_diff = df["Time"].diff().dropna()

    sampling_summary.append({
        "File": file.name,
        "Minimum Interval": time_diff.min(),
        "Median Interval": time_diff.median(),
        "Most Common Interval": time_diff.mode().iloc[0],
        "Maximum Interval": time_diff.max(),
        "Unique Intervals": time_diff.nunique()
    })

sampling_summary_df = pd.DataFrame(sampling_summary)

sampling_summary_df

,File,Minimum Interval,Median Interval,Most Common Interval,Maximum Interval,Unique Intervals
0,acv_case_01.xlsx,0 days 00:00:27,0 days 00:00:30,0 days 00:00:30,0 days 09:39:30,15
1,acv_case_02.xlsx,0 days 00:00:27,0 days 00:00:30,0 days 00:00:30,0 days 05:58:30,20
2,acv_case_03.xlsx,0 days 00:00:30,0 days 00:00:30,0 days 00:00:30,0 days 09:29:30,15
3,acv_case_04.xlsx,0 days 00:00:01,0 days 00:00:10,0 days 00:00:10,0 days 10:52:10,18
4,acv_case_05.xlsx,0 days 00:00:30,0 days 00:00:30,0 days 00:00:30,0 days 11:20:30,17
5,acv_case_06.xlsx,0 days 00:00:30,0 days 00:00:30,0 days 00:00:30,0 days 08:00:00,17
6,acv_test_case.xlsx,0 days 00:00:30,0 days 00:00:30,0 days 00:00:30,0 days 06:10:30,17


### 11.2.1 Observation

Cases 01, 02, 03, 05, 06, and the test case have a median and most common sampling interval of 30 seconds, confirming that the standard ACV telemetry is normally sampled every 30 seconds.

Cases 01 and 02 contain a small number of intervals slightly shorter than 30 seconds, with a minimum of 27 seconds, indicating minor timestamp irregularities.

However, all files also contain substantially larger timestamp gaps. For example, the maximum interval reaches 11 hours 20 minutes 30 seconds in Case 05 and 6 hours 10 minutes 30 seconds in the test case. Therefore, the recordings should not be assumed to represent uninterrupted telemetry over their entire date ranges.

Case 04 differs from the standard cases, with a median and most common sampling interval of 10 seconds. This further confirms that Case 04 was collected under a different telemetry configuration and should not be assumed to follow the same temporal structure as the standard-schema cases.

### 11.3 Timestamp Ordering

Verify that timestamps are monotonically increasing within each file. Correct temporal ordering is required for later calculations involving trends, transitions, rolling windows, and durations.

In [30]:
for file in all_files:
    df = pd.read_excel(file)

    print(
        f"{file.name}: "
        f"{df['Time'].is_monotonic_increasing}"
    )

acv_case_01.xlsx: True
acv_case_02.xlsx: True
acv_case_03.xlsx: True
acv_case_04.xlsx: True
acv_case_05.xlsx: True
acv_case_06.xlsx: True
acv_test_case.xlsx: True


### 11.3.1 Observation

Timestamps are monotonically increasing in all six training cases and the test case. Therefore, the observations are already arranged in chronological order and do not require sorting before temporal analysis.

However, chronological ordering does not imply continuous sampling. As shown previously, several files contain large gaps between consecutive timestamps, which must be considered when constructing temporal features.

### 11.4 Large Timestamp Gaps

Although timestamps are correctly ordered, the sampling-interval analysis identified periods where consecutive observations are separated by substantially longer durations than the normal sampling interval.

The number of intervals exceeding the expected sampling interval is calculated to quantify these interruptions in telemetry collection.

In [31]:
timestamp_gap_summary = []

for file in all_files:
    df = pd.read_excel(file)

    time_diff = df["Time"].diff().dropna()

    # Case 04 normally samples every 10 seconds;
    # the standard cases normally sample every 30 seconds.
    expected_interval = (
        pd.Timedelta(seconds=10)
        if file.name == "acv_case_04.xlsx"
        else pd.Timedelta(seconds=30)
    )

    larger_gaps = time_diff[time_diff > expected_interval]

    timestamp_gap_summary.append({
        "File": file.name,
        "Expected Interval": expected_interval,
        "Intervals > Expected": len(larger_gaps),
        "Percentage > Expected": (
            len(larger_gaps) / len(time_diff) * 100
        ),
        "Largest Gap": (
            larger_gaps.max()
            if len(larger_gaps) > 0
            else pd.Timedelta(0)
        )
    })

timestamp_gap_df = pd.DataFrame(timestamp_gap_summary)

timestamp_gap_df

,File,Expected Interval,Intervals > Expected,Percentage > Expected,Largest Gap
0,acv_case_01.xlsx,0 days 00:00:30,236,3.372392,0 days 09:39:30
1,acv_case_02.xlsx,0 days 00:00:30,142,1.545831,0 days 05:58:30
2,acv_case_03.xlsx,0 days 00:00:30,159,1.913588,0 days 09:29:30
3,acv_case_04.xlsx,0 days 00:00:10,36,0.161718,0 days 10:52:10
4,acv_case_05.xlsx,0 days 00:00:30,119,1.707072,0 days 11:20:30
5,acv_case_06.xlsx,0 days 00:00:30,119,3.648069,0 days 08:00:00
6,acv_test_case.xlsx,0 days 00:00:30,193,2.125317,0 days 06:10:30


### 11.4.1 Observation

Most consecutive observations follow the expected sampling interval. In the standard-schema files, approximately 1.5–3.6% of timestamp intervals exceed the expected 30-second interval, while approximately 2.1% of intervals in the test case exceed 30 seconds.

Although these interruptions represent a relatively small proportion of consecutive observations, some individual gaps are several hours long. Therefore, later temporal feature engineering should account for elapsed time rather than assuming that every pair of adjacent rows represents consecutive 30-second measurements.

Case 04 again exhibits a different temporal structure. Only approximately 0.16% of its intervals exceed its expected 10-second sampling interval, although several large interruptions are still present.

## 12. Telemetry Parameter Characteristics

The standard ACV schema contains eight telemetry parameters for each car. Before feature engineering, the value distributions and data types of these parameters are inspected to distinguish continuous measurements from discrete operating states or status indicators.

Understanding the characteristics of each parameter will determine how the signals should later be cleaned, represented, and transformed into fault-detection features.

In [32]:
case_03 = pd.read_excel(TRAIN_DIR / "acv_case_03.xlsx")

case_03_telemetry_cols = [
    col for col in case_03.columns
    if col not in metadata_columns and " - " in col
]

parameter_characteristics = []

for parameter in sorted({
    col.split(" - ", 1)[1]
    for col in case_03_telemetry_cols
}):
    parameter_cols = [
        col for col in case_03_telemetry_cols
        if col.split(" - ", 1)[1] == parameter
    ]

    values = case_03[parameter_cols].stack()

    parameter_characteristics.append({
        "Parameter": parameter,
        "Data Type": str(values.dtype),
        "Unique Values": values.nunique(),
        "Minimum": values.min(),
        "Maximum": values.max()
    })

parameter_characteristics_df = pd.DataFrame(
    parameter_characteristics
)

parameter_characteristics_df

,Parameter,Data Type,Unique Values,Minimum,Maximum
0,ACV Control Temperature (Cooling),float64,9,0.0,27.0
1,ACV Control Temperature (Heating),int64,2,0,20
2,ACV Information Valid,str,2,Invalid,Valid
3,ACV Running Mode,str,6,Automatic Cooling,Ventilation
4,ACV Setting Mode,str,3,Centralized Control,Manual Control
5,Indoor Average Temperature,float64,21,0.0,30.5
6,Load Halved,str,1,Normal,Normal
7,Outdoor Average Temperature,float64,32,0.0,38.5


### 12.1 Observation

The standard ACV telemetry contains a mixture of numerical measurements and categorical operating-state variables.

The temperature-related parameters are numerical, although several take only a limited number of distinct values. In particular, the heating and cooling control temperatures appear to operate at discrete setpoints rather than behaving as continuously varying sensor measurements.

`ACV Information Valid`, `ACV Running Mode`, `ACV Setting Mode`, and `Load Halved` are categorical status or operating-state variables. In Case 03, `Load Halved` contains only the value `Normal`, while the other categorical parameters contain multiple states.

These characteristics indicate that numerical and categorical telemetry should not necessarily be processed using the same feature-engineering strategy. However, Case 03 alone is insufficient to determine the complete set of possible states, so the parameter characteristics should be compared across the remaining standard-schema training cases.

### 12.2 Categorical States Across Training Cases

Case 03 provides an initial view of the telemetry characteristics, but a single case may not contain every possible operating state.

The categorical parameters are therefore inspected across all standard-schema training cases to identify the complete set of observed states and determine whether their representations are consistent between cases.

In [33]:
categorical_parameters = [
    "ACV Information Valid",
    "ACV Running Mode",
    "ACV Setting Mode",
    "Load Halved"
]

standard_train_files = [
    TRAIN_DIR / "acv_case_01.xlsx",
    TRAIN_DIR / "acv_case_02.xlsx",
    TRAIN_DIR / "acv_case_03.xlsx",
    TRAIN_DIR / "acv_case_05.xlsx",
    TRAIN_DIR / "acv_case_06.xlsx"
]

categorical_state_summary = []

for file in standard_train_files:
    df = pd.read_excel(file)

    for parameter in categorical_parameters:
        parameter_cols = [
            col for col in df.columns
            if " - " in col
            and col.split(" - ", 1)[1] == parameter
        ]

        values = (
            df[parameter_cols]
            .stack()
            .dropna()
        )

        categorical_state_summary.append({
            "File": file.name,
            "Parameter": parameter,
            "Unique Values": values.nunique(),
            "Observed States": sorted(values.unique().tolist())
        })

categorical_state_df = pd.DataFrame(
    categorical_state_summary
)

categorical_state_df

,File,Parameter,Unique Values,Observed States
0,acv_case_01.xlsx,ACV Information Valid,2,"[Invalid, Valid]"
1,acv_case_01.xlsx,ACV Running Mode,5,"[Automatic Cooling, Emergency Ventilation, Ful..."
2,acv_case_01.xlsx,ACV Setting Mode,3,"[Centralized Control, Invalid, Manual Control]"
3,acv_case_01.xlsx,Load Halved,1,[Normal]
4,acv_case_02.xlsx,ACV Information Valid,2,"[Invalid, Valid]"
5,acv_case_02.xlsx,ACV Running Mode,4,"[Automatic Cooling, Emergency Ventilation, Inv..."
6,acv_case_02.xlsx,ACV Setting Mode,3,"[Centralized Control, Invalid, Manual Control]"
7,acv_case_02.xlsx,Load Halved,1,[Normal]
8,acv_case_03.xlsx,ACV Information Valid,2,"[Invalid, Valid]"
9,acv_case_03.xlsx,ACV Running Mode,6,"[Automatic Cooling, Emergency Ventilation, Hal..."


### 12.2.1 Observation

The categorical telemetry parameters exhibit different levels of variation across the standard training cases.

`Load Halved` is constant across all five cases, with only the state `Normal` observed. Based on the training data inspected so far, this parameter therefore provides no variation for distinguishing operating behaviour.

`ACV Information Valid` contains both `Valid` and `Invalid` states in Cases 01–03, while only `Valid` is observed in Cases 05–06. Similarly, `ACV Setting Mode` includes an `Invalid` state in Cases 01–03 but only the two control modes in Cases 05–06.

`ACV Running Mode` contains the greatest categorical variation, with between four and six states observed depending on the training case.

The complete state names should be inspected before determining whether categorical representations are fully consistent across cases.

### 12.3 Complete Categorical State Set

The table display truncates some categorical state lists. The complete set of states observed across all standard training cases is therefore extracted for each categorical parameter.

In [34]:
for parameter in categorical_parameters:
    all_states = set()

    for file in standard_train_files:
        df = pd.read_excel(file)

        parameter_cols = [
            col for col in df.columns
            if " - " in col
            and col.split(" - ", 1)[1] == parameter
        ]

        values = (
            df[parameter_cols]
            .stack()
            .dropna()
        )

        all_states.update(values.unique())

    print(f"\n{parameter}")
    
    for state in sorted(all_states):
        print(f"  - {state}")


ACV Information Valid
  - Invalid
  - Valid

ACV Running Mode
  - Automatic Cooling
  - Emergency Ventilation
  - Full Cooling
  - Half Cooling
  - Invalid
  - Stop
  - Ventilation

ACV Setting Mode
  - Centralized Control
  - Invalid
  - Manual Control

Load Halved
  - Normal


### 12.3.1 Observation

The categorical states are represented consistently across the standard training cases.

`ACV Information Valid` indicates whether the ACV information is valid or invalid. `ACV Running Mode` contains seven observed operating states, including cooling, ventilation, stop, and invalid states. `ACV Setting Mode` contains centralized control, manual control, and invalid states.

`Load Halved` contains only the value `Normal` across all standard training cases and therefore exhibits no variation in the available training data.

The presence of explicit `Invalid` states in several categorical parameters also suggests that some numerical values, particularly zero-valued temperatures, may correspond to periods of invalid telemetry rather than physical measurements.

### 12.4 Relationship Between Information Validity and Temperature Values

Several numerical temperature parameters contain zero values, while the categorical telemetry includes an explicit `ACV Information Valid` state.

To determine whether zero-valued temperatures may represent invalid telemetry rather than physical measurements, temperature values are compared between valid and invalid information states.

In [35]:
temperature_parameters = [
    "ACV Control Temperature (Cooling)",
    "ACV Control Temperature (Heating)",
    "Indoor Average Temperature",
    "Outdoor Average Temperature"
]

validity_comparison = []

for car in cars:
    validity_col = f"{car} - ACV Information Valid"

    for parameter in temperature_parameters:
        parameter_col = f"{car} - {parameter}"

        for validity_state in ["Valid", "Invalid"]:
            values = case_03.loc[
                case_03[validity_col] == validity_state,
                parameter_col
            ].dropna()

            validity_comparison.append({
                "Car": car,
                "Parameter": parameter,
                "Validity": validity_state,
                "Count": len(values),
                "Zero (%)": (
                    (values == 0).mean() * 100
                    if len(values) > 0 else np.nan
                )
            })

validity_comparison_df = pd.DataFrame(validity_comparison)

validity_comparison_df.groupby(
    ["Parameter", "Validity"]
)[["Count", "Zero (%)"]].mean().round(2)

Count  Zero (%)
Parameter                         Validity                   
ACV Control Temperature (Cooling) Invalid      0.12     100.0
                                  Valid     8309.88       0.0
ACV Control Temperature (Heating) Invalid      0.12     100.0
                                  Valid     8309.88       0.0
Indoor Average Temperature        Invalid      0.12     100.0
                                  Valid     8309.88       0.0
Outdoor Average Temperature       Invalid      0.12     100.0
                                  Valid     8309.88       0.0

### 12.4.1 Observation

In Case 03, all observed temperature values associated with an `Invalid` ACV information state are zero, while no zero-valued temperatures occur when the corresponding ACV information state is `Valid`.

This indicates that zero-valued temperatures in Case 03 are associated with invalid telemetry rather than normal temperature measurements. However, the number of invalid observations in this case is very small, so this relationship should not be assumed to hold universally without further validation.

The `ACV Information Valid` field should therefore be retained during preprocessing so that invalid telemetry can be identified explicitly rather than interpreting zero values as ordinary physical measurements.